In [1]:
import torch
import arviz as az
from scipy.optimize import curve_fit
from bokeh.plotting import figure, show,output_file, save
from bokeh.transform import factor_cmap, factor_mark
from bokeh.palettes import Spectral
from bokeh.models import Slope, Div
from bokeh.io import curdoc,output_notebook,export_png
from bokeh.layouts import column,gridplot
import numpy as np
from sklearn.metrics import r2_score
#from seaborn import clustermap*
from bokeh.models import Band, ColumnDataSource
import pandas as pd
from Plotting import *
TOOLS="hover,crosshair,pan,wheel_zoom,zoom_in,zoom_out,box_zoom,undo,redo,reset,tap,save,box_select,poly_select,lasso_select,examine,help"

In [2]:
PM25=pd.read_excel("../../Data_and_Preprocessing/Pre_flow_cal.xlsx")
train,test=PM25.loc[PM25.label==0,:],PM25.loc[PM25.label==1,:]

In [3]:
gpr=torch.load("../models/PM2.5_flow",weights_only=False,map_location='cuda')
gpr_bc=torch.load("../models/bc_flow",weights_only=False,map_location='cuda')

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
def gaussian_plot_flow(data,model,input_x="flow"):
    linmod=torch.linspace(data[input_x].min(),data[input_x].max(),500).to(device)
    mean,covarience=model(linmod,full_cov=True)
    linmod_np=linmod.cpu().detach().numpy()
    model_np,model_std_np=mean.cpu().detach().numpy().copy(),covarience.diag().sqrt().cpu() .detach().numpy().copy()
    lower = model_np - model_std_np
    upper = model_np + model_std_np
    data2=pd.DataFrame([linmod_np,model_np,lower,upper],index=["flow","y","lower","upper"]).T.set_index("flow")
    
    data1 = ColumnDataSource(data2.reset_index())
    return data1,data2

In [6]:
plot_data,model_pm=gaussian_plot_flow(train,gpr)
plot_data,model_bc=gaussian_plot_flow(train,gpr_bc)

In [7]:
data_plot_pm,data_mean_pm=roll_week(train,"pm_final",input_x="flow",step=0.1)
data_plot_sv,data_mean_sv=roll_week(train,"sv_final",input_x="flow",step=0.1)

In [8]:
data_plot_pm_test,data_mean_pm_test=roll_week(test,"pm_final",input_x="flow",step=0.1)
data_plot_sv_test,data_mean_sv_test=roll_week(test,"sv_final",input_x="flow",step=0.1)


In [9]:
r2_flow_pm,r2_flow_sv=r2_score(data_mean_pm.y,model_pm.y),r2_score(data_mean_sv.y,model_bc.y)


In [10]:
data_mean_pm

,y,lower,upper
3.47000,24.884143,7.846128,41.922158
3.47206,24.884143,7.846128,41.922158
3.47412,24.884143,7.846128,41.922158
3.47618,24.884143,7.846128,41.922158
3.47824,24.884143,7.846128,41.922158
...,...,...,...
4.48976,15.840000,1.507909,30.172091
4.49182,15.840000,1.507909,30.172091
4.49388,15.840000,1.507909,30.172091
4.49594,15.840000,1.507909,30.172091


In [11]:
output_notebook()
p = figure(x_range=(3.6,4.3),y_range=(0,60));
p.title.text = r"$$ PM_{2.5}$$ calibration of flow";
p.xgrid.grid_line_color=None;
p.ygrid.grid_line_alpha=0.5;

p.scatter(train.flow, y=train.pm_final, color="blue", marker="dot", size=20, alpha=0.4,legend_label="raw points");
band = Band(base="index", lower="lower", upper="upper",source=data_plot_pm, fill_color="red", line_color="black",fill_alpha=0.2);
band1 = Band(base="index", lower="lower", upper="upper",source=plot_data,fill_alpha=0.5, fill_color="blue", line_color="black");

p.add_layout(band);
p.add_layout(band1);
p.line(data_mean_pm.index, data_mean_pm.y, line_width=3,color="green",legend_label="Train roll mean R2 "+str(round(r2_flow_pm,2)));
p.line(data_mean_sv_test.index, data_mean_sv_test.y, line_width=3,color="orange",legend_label="test roll mean R2 "+str(round(r2_flow_sv,2)));

p.line(model_pm.index, model_pm.y, line_width=3,color="red",legend_label="Gaussion model");
p.xaxis.axis_label = r'$$\frac{L}{s}$$';
p.yaxis.axis_label = r'$$\frac{\mu g}{m^3} $$';
p.yaxis.axis_label_orientation  = 0

show(p);

Loading BokehJS ...

In [ ]:
A